In [ ]:
print("Task Scheduler & Job Runner")
print("Project setup successful!")

Task Scheduler & Job Runner
Project setup successful!


In [ ]:
!pip install flask apscheduler flask-cors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 4.9 MB/s eta 0:00:00


In [ ]:
from flask import Flask, jsonify
from apscheduler.schedulers.background import BackgroundScheduler
from datetime import datetime

app = Flask(__name__)

scheduler = BackgroundScheduler()
scheduler.start()

tasks = []

def run_task(task_name):
    print(f"✅ Running task: {task_name}")
    print(f"⏰ Time: {datetime.now()}")

@app.route("/")
def home():
    return "Task Scheduler & Job Runner is running!"

@app.route("/tasks")
def get_tasks():
    return jsonify(tasks)

@app.route("/add/<task_name>")
def add_task(task_name):
    job = scheduler.add_job(
        run_task,
        "interval",
        minutes=1,
        args=[task_name]
    )

    task = {
        "id": job.id,
        "name": task_name,
        "status": "Scheduled",
        "created": str(datetime.now())
    }

    tasks.append(task)

    return jsonify({
        "message": "Task scheduled successfully!",
        "task": task
    })

print("🚀 Task Scheduler started!")

🚀 Task Scheduler started!


In [ ]:
# Add a test task
result = add_task("My First Job")
print(result.get_json())

RuntimeError: Working outside of application context.

This typically means that you attempted to use functionality that needed
the current application. To solve this, set up an application context
with app.app_context(). See the documentation for more information.

In [ ]:
with app.app_context():
    result = add_task("My First Job")
    print(result.get_json())

{'message': 'Task scheduled successfully!', 'task': {'created': '2026-10-06 16:30:29.223667', 'id': 'a08106cb35f3483fa60bdd2795006510', 'name': 'My First Job', 'status': 'Scheduled'}}


In [ ]:
print("📋 Scheduled Tasks")
print("-" * 40)

for task in tasks:
    print("Task ID:", task["id"])
    print("Task Name:", task["name"])
    print("Status:", task["status"])
    print("Created:", task["created"])
    print("-" * 40)

📋 Scheduled Tasks
----------------------------------------
Task ID: 5ca405a6870a4806bad6e49538bce1dc
Task Name: My First Job
Status: Scheduled
Created: 2026-10-06 16:28:24.079797
----------------------------------------
Task ID: a08106cb35f3483fa60bdd2795006510
Task Name: My First Job
Status: Scheduled
Created: 2026-10-06 16:30:29.223667
----------------------------------------


In [ ]:
# Execution history
execution_history = []

def run_task_with_history(task_name):
    start_time = datetime.now()

    print(f"▶️ Running: {task_name}")

    # Simulated task work
    result = "Success"

    end_time = datetime.now()

    execution_history.append({
        "task": task_name,
        "status": result,
        "started": str(start_time),
        "finished": str(end_time)
    })

    print(f"✅ {task_name} completed successfully!")
    print(f"⏰ Started: {start_time}")
    print(f"⏰ Finished: {end_time}")


print("📊 Execution monitoring is ready!")

📊 Execution monitoring is ready!


In [ ]:
# Run a task manually
run_task_with_history("My First Job")

print("\n📊 Execution History")
print("-" * 50)

for record in execution_history:
    print("Task:", record["task"])
    print("Status:", record["status"])
    print("Started:", record["started"])
    print("Finished:", record["finished"])
    print("-" * 50)

▶️ Running: My First Job
✅ My First Job completed successfully!
⏰ Started: 2026-10-06 16:32:31.627889
⏰ Finished: 2026-10-06 16:32:31.627944

📊 Execution History
--------------------------------------------------
Task: My First Job
Status: Success
Started: 2026-10-06 16:32:31.627889
Finished: 2026-10-06 16:32:31.627944
--------------------------------------------------


In [ ]:
from flask import render_template_string

dashboard_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Task Scheduler & Job Runner</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f4f6f8;
            margin: 0;
            padding: 30px;
        }

        .container {
            max-width: 900px;
            margin: auto;
        }

        h1 {
            text-align: center;
        }

        .card {
            background: white;
            padding: 20px;
            margin: 15px 0;
            border-radius: 12px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }

        input, button {
            padding: 12px;
            margin: 5px;
            border-radius: 6px;
            border: 1px solid #ccc;
        }

        button {
            cursor: pointer;
            background: #222;
            color: white;
        }

        .status {
            font-weight: bold;
        }
    </style>
</head>

<body>

<div class="container">

    <h1>⏰ Task Scheduler & Job Runner</h1>

    <div class="card">
        <h2>Add New Task</h2>

        <input id="taskName" placeholder="Enter task name">

        <button onclick="addTask()">
            ➕ Schedule Task
        </button>
    </div>

    <div class="card">
        <h2>📋 Scheduled Tasks</h2>

        <div id="taskList">
            Loading...
        </div>
    </div>

</div>

<script>

async function loadTasks() {

    const response = await fetch("/tasks");

    const tasks = await response.json();

    let html = "";

    if (tasks.length === 0) {
        html = "<p>No tasks scheduled.</p>";
    }

    tasks.forEach(task => {

        html += `
        <div style="border:1px solid #ddd;padding:15px;margin:10px 0;border-radius:8px">

            <h3>${task.name}</h3>

            <p>
                Status:
                <span class="status">
                    🟢 ${task.status}
                </span>
            </p>

            <p>Created: ${task.created}</p>

        </div>
        `;

    });

    document.getElementById("taskList").innerHTML = html;
}


async function addTask() {

    const name = document.getElementById("taskName").value;

    if (!name) {
        alert("Please enter a task name");
        return;
    }

    await fetch("/add/" + encodeURIComponent(name));

    document.getElementById("taskName").value = "";

    loadTasks();
}


loadTasks();

</script>

</body>
</html>
"""

@app.route("/dashboard")
def dashboard():
    return render_template_string(dashboard_html)

print("🌐 Dashboard code is ready!")

🌐 Dashboard code is ready!


In [ ]:
import threading
import time

def start_server():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=start_server)
server_thread.daemon = True
server_thread.start()

time.sleep(3)

print("🚀 Flask server started!")
print("📱 Dashboard: http://127.0.0.1:5000/dashboard")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🚀 Flask server started!
📱 Dashboard: http://127.0.0.1:5000/dashboard


In [ ]:
!pip install pyngrok -q

✅ Running task: My First Job
⏰ Time: 2026-10-06 16:35:24.081946
✅ Running task: My First Job
⏰ Time: 2026-10-06 16:35:29.227356


In [ ]:
3KKU3LbrYRSsjH6LWrsYJBzaqiN_2UMMTtXXsQSgTPCF41fRn

SyntaxError: invalid decimal literal (1546569851.py, line 1)

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3KKU3LbrYRSsjH6LWrsYJBzaqiN_2UMMTtXXsQSgTPCF41fRn")

public_url = ngrok.connect(5000)

print("🌐 PUBLIC DASHBOARD LINK:")
print(public_url)

🌐 PUBLIC DASHBOARD LINK:
NgrokTunnel: "https://spider-dreamt-plated.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
from apscheduler.triggers.cron import CronTrigger

def create_cron_task(task_name, hour, minute):
    job_id = "cron_" + task_name.replace(" ", "_")

    job = scheduler.add_job(
        run_task_with_history,
        CronTrigger(hour=hour, minute=minute),
        args=[task_name],
        id=job_id,
        replace_existing=True
    )

    task = {
        "id": job.id,
        "name": task_name,
        "schedule": f"Daily at {hour:02d}:{minute:02d}",
        "status": "Scheduled",
        "created": str(datetime.now())
    }

    tasks.append(task)

    return task


print("⏰ Cron Job Management is ready!")

⏰ Cron Job Management is ready!


In [ ]:
cron_task = create_cron_task(
    "Daily Backup",
    22,
    10
)

print("✅ Cron job created!")
print("Task:", cron_task["name"])
print("Schedule:", cron_task["schedule"])
print("Status:", cron_task["status"])

✅ Cron job created!
Task: Daily Backup
Schedule: Daily at 22:10
Status: Scheduled


In [ ]:
# Test the Job Runner

run_task_with_history("Daily Backup")

print("\n📊 EXECUTION MONITORING")
print("-" * 50)

for record in execution_history:
    print("Task:", record["task"])
    print("Status:", record["status"])
    print("Started:", record["started"])
    print("Finished:", record["finished"])
    print("-" * 50)

▶️ Running: Daily Backup
✅ Daily Backup completed successfully!
⏰ Started: 2026-10-06 17:07:06.091541
⏰ Finished: 2026-10-06 17:07:06.091597

📊 EXECUTION MONITORING
--------------------------------------------------
Task: My First Job
Status: Success
Started: 2026-10-06 16:32:31.627889
Finished: 2026-10-06 16:32:31.627944
--------------------------------------------------
Task: Daily Backup
Status: Success
Started: 2026-10-06 17:07:06.091541
Finished: 2026-10-06 17:07:06.091597
--------------------------------------------------


In [ ]:
def pause_task(task_id):
    try:
        scheduler.pause_job(task_id)

        for task in tasks:
            if task["id"] == task_id:
                task["status"] = "Paused"

        return f"⏸️ Task {task_id} paused successfully."

    except Exception as e:
        return f"❌ Error: {e}"


def resume_task(task_id):
    try:
        scheduler.resume_job(task_id)

        for task in tasks:
            if task["id"] == task_id:
                task["status"] = "Scheduled"

        return f"▶️ Task {task_id} resumed successfully."

    except Exception as e:
        return f"❌ Error: {e}"


def delete_task(task_id):
    try:
        scheduler.remove_job(task_id)

        for task in tasks:
            if task["id"] == task_id:
                task["status"] = "Deleted"

        return f"🗑️ Task {task_id} deleted successfully."

    except Exception as e:
        return f"❌ Error: {e}"


print("⚙️ Job Management is ready!")

⚙️ Job Management is ready!


In [ ]:
# Get the first scheduled job
job = scheduler.get_jobs()[0]

print("🔎 Testing Job Management")
print("Job ID:", job.id)

# Pause
print(pause_task(job.id))

# Resume
print(resume_task(job.id))

print("✅ Pause and Resume test completed!")

🔎 Testing Job Management
Job ID: 5ca405a6870a4806bad6e49538bce1dc
⏸️ Task 5ca405a6870a4806bad6e49538bce1dc paused successfully.
▶️ Task 5ca405a6870a4806bad6e49538bce1dc resumed successfully.
✅ Pause and Resume test completed!


In [ ]:
# Create a temporary test job

test_job = scheduler.add_job(
    run_task_with_history,
    "interval",
    minutes=10,
    args=["Temporary Test Job"]
)

print("🆕 Test job created")
print("Job ID:", test_job.id)

# Delete the test job
print(delete_task(test_job.id))

print("✅ Delete Job test completed!")

🆕 Test job created
Job ID: 2caf6d2797b2450388f75a3ea0f2a0eb
🗑️ Task 2caf6d2797b2450388f75a3ea0f2a0eb deleted successfully.
✅ Delete Job test completed!


In [ ]:
import sqlite3

# Create database
conn = sqlite3.connect("tasks.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tasks (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    schedule TEXT,
    status TEXT,
    created TEXT
)
""")

conn.commit()

print("🗄️ SQLite database created successfully!")
print("📁 Database file: tasks.db")

🗄️ SQLite database created successfully!
📁 Database file: tasks.db


In [ ]:
def save_task_to_database(task):
    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute("""
    INSERT OR REPLACE INTO tasks
    (id, name, schedule, status, created)
    VALUES (?, ?, ?, ?, ?)
    """, (
        task["id"],
        task["name"],
        task.get("schedule", "Every 1 minute"),
        task["status"],
        task["created"]
    ))

    conn.commit()
    conn.close()

    print("💾 Task saved:", task["name"])


# Save all existing tasks
for task in tasks:
    save_task_to_database(task)

print("✅ All current tasks saved to SQLite!")

✅ Running task: Daily Report
⏰ Time: 2026-10-06 17:09:54.057573
💾 Task saved: My First Job
💾 Task saved: My First Job
💾 Task saved: Daily Report
💾 Task saved: Daily Backup
✅ All current tasks saved to SQLite!


In [ ]:
def load_tasks_from_database():
    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM tasks")
    rows = cursor.fetchall()

    conn.close()

    print("📂 Saved Tasks")
    print("-" * 50)

    for row in rows:
        print("Task ID:", row[0])
        print("Task Name:", row[1])
        print("Schedule:", row[2])
        print("Status:", row[3])
        print("Created:", row[4])
        print("-" * 50)

    return rows


saved_tasks = load_tasks_from_database()

print("✅ Database loading completed!")

📂 Saved Tasks
--------------------------------------------------
Task ID: 5ca405a6870a4806bad6e49538bce1dc
Task Name: My First Job
Schedule: Every 1 minute
Status: Scheduled
Created: 2026-10-06 16:28:24.079797
--------------------------------------------------
Task ID: a08106cb35f3483fa60bdd2795006510
Task Name: My First Job
Schedule: Every 1 minute
Status: Scheduled
Created: 2026-10-06 16:30:29.223667
--------------------------------------------------
Task ID: 3ae9970c44d2455abcdcbcc712e44315
Task Name: Daily Report
Schedule: Every 1 minute
Status: Scheduled
Created: 2026-10-06 17:01:54.057230
--------------------------------------------------
Task ID: cron_Daily_Backup
Task Name: Daily Backup
Schedule: Daily at 22:10
Status: Scheduled
Created: 2026-10-06 17:06:18.007982
--------------------------------------------------
✅ Database loading completed!


In [ ]:
# Stop the current Flask server
try:
    scheduler.shutdown(wait=False)
except:
    pass

print("🛑 Old server setup stopped.")

🛑 Old server setup stopped.


In [ ]:
⁸from flask import jsonify

@app.route("/history")
def get_history():
    return jsonify(execution_history)

print("📊 Execution History API added successfully!")

AssertionError: The setup method 'route' can no longer be called on the application. It has already handled its first request, any changes will not be applied consistently.
Make sure all imports, decorators, functions, etc. needed to set up the application are done before running it.

In [ ]:
!pip install flask apscheduler flask-cors pyngrok -q

from flask import Flask, jsonify, request, render_template_string
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.cron import CronTrigger
from datetime import datetime
import sqlite3

app = Flask(__name__)

scheduler = BackgroundScheduler()
scheduler.start()

execution_history = []


# ---------------- DATABASE ----------------

def init_database():
    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS tasks (
        id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        schedule TEXT,
        status TEXT,
        created TEXT
    )
    """)

    conn.commit()
    conn.close()


init_database()


# ---------------- JOB RUNNER ----------------

def run_task(task_name):
    start = datetime.now()

    print(f"▶️ Running: {task_name}")

    status = "Success"

    finish = datetime.now()

    execution_history.append({
        "task": task_name,
        "status": status,
        "started": str(start),
        "finished": str(finish)
    })

    print(f"✅ {task_name} completed!")


# ---------------- ADD CRON JOB ----------------

def create_cron_task(task_name, hour, minute):

    job_id = "cron_" + task_name.replace(" ", "_")

    job = scheduler.add_job(
        run_task,
        CronTrigger(hour=hour, minute=minute),
        args=[task_name],
        id=job_id,
        replace_existing=True
    )

    task = {
        "id": job.id,
        "name": task_name,
        "schedule": f"Daily at {hour:02d}:{minute:02d}",
        "status": "Scheduled",
        "created": str(datetime.now())
    }

    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute("""
    INSERT OR REPLACE INTO tasks
    (id, name, schedule, status, created)
    VALUES (?, ?, ?, ?, ?)
    """, (
        task["id"],
        task["name"],
        task["schedule"],
        task["status"],
        task["created"]
    ))

    conn.commit()
    conn.close()

    return task


# ---------------- DASHBOARD ----------------

HTML = """
<!DOCTYPE html>
<html>
<head>

<title>Task Scheduler & Job Runner</title>

<meta name="viewport" content="width=device-width, initial-scale=1">

<style>

body {
    font-family: Arial;
    background: #f4f6f8;
    padding: 20px;
}

.container {
    max-width: 800px;
    margin: auto;
}

.card {
    background: white;
    padding: 20px;
    margin-bottom: 20px;
    border-radius: 12px;
}

input, button {
    padding: 10px;
    margin: 5px;
}

button {
    background: black;
    color: white;
    border: none;
    border-radius: 6px;
}

.task {
    border: 1px solid #ddd;
    padding: 12px;
    margin: 10px 0;
    border-radius: 8px;
}

</style>

</head>

<body>

<div class="container">

<h1>⏰ Task Scheduler & Job Runner</h1>

<div class="card">

<h2>➕ Create Cron Job</h2>

<input id="name" placeholder="Task name">

<br>

<label>Hour:</label>
<input id="hour" type="number" min="0" max="23" value="22">

<label>Minute:</label>
<input id="minute" type="number" min="0" max="59" value="30">

<br>

<button onclick="createTask()">
Create Cron Job
</button>

</div>


<div class="card">

<h2>📋 Scheduled Tasks</h2>

<div id="tasks">
Loading...
</div>

</div>


<div class="card">

<h2>📊 Execution History</h2>

<div id="history">
Loading...
</div>

</div>

</div>


<script>

async function createTask() {

    let name = document.getElementById("name").value;

    let hour = document.getElementById("hour").value;

    let minute = document.getElementById("minute").value;

    if (!name) {
        alert("Enter task name");
        return;
    }

    await fetch(
        `/create?name=${encodeURIComponent(name)}&hour=${hour}&minute=${minute}`
    );

    document.getElementById("name").value = "";

    loadTasks();
}


async function loadTasks() {

    let response = await fetch("/tasks");

    let tasks = await response.json();

    let html = "";

    if (tasks.length === 0) {
        html = "<p>No tasks yet.</p>";
    }

    tasks.forEach(task => {

        html += `
        <div class="task">

        <b>${task.name}</b>

        <p>⏰ ${task.schedule}</p>

        <p>🟢 ${task.status}</p>

        <small>${task.created}</small>

        </div>
        `;

    });

    document.getElementById("tasks").innerHTML = html;
}


async function loadHistory() {

    let response = await fetch("/history");

    let history = await response.json();

    let html = "";

    if (history.length === 0) {
        html = "<p>No executions yet.</p>";
    }

    history.forEach(item => {

        html += `
        <div class="task">

        <b>${item.task}</b>

        <p>✅ ${item.status}</p>

        <small>
        Started: ${item.started}<br>
        Finished: ${item.finished}
        </small>

        </div>
        `;

    });

    document.getElementById("history").innerHTML = html;
}


async function refresh() {

    loadTasks();

    loadHistory();

}


refresh();

</script>

</body>
</html>
"""


# ---------------- FLASK ROUTES ----------------

@app.route("/")
def home():
    return "Task Scheduler & Job Runner is running!"


@app.route("/dashboard")
def dashboard():
    return render_template_string(HTML)


@app.route("/create")
def create():

    name = request.args.get("name")

    hour = int(request.args.get("hour"))

    minute = int(request.args.get("minute"))

    task = create_cron_task(name, hour, minute)

    return jsonify(task)


@app.route("/tasks")
def tasks_api():

    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM tasks")

    rows = cursor.fetchall()

    conn.close()

    result = []

    for row in rows:

        result.append({
            "id": row[0],
            "name": row[1],
            "schedule": row[2],
            "status": row[3],
            "created": row[4]
        })

    return jsonify(result)


@app.route("/history")
def history_api():

    return jsonify(execution_history)


print("🚀 TASK SCHEDULER PROJECT READY!")
print("✅ Flask")
print("✅ APScheduler")
print("✅ Cron Jobs")
print("✅ Job Runner")
print("✅ SQLite Database")
print("✅ Execution Monitoring")
print("✅ Web Dashboard")

🚀 TASK SCHEDULER PROJECT READY!
✅ Flask
✅ APScheduler
✅ Cron Jobs
✅ Job Runner
✅ SQLite Database
✅ Execution Monitoring
✅ Web Dashboard


In [ ]:
import threading
import time

def start_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

server_thread = threading.Thread(target=start_flask)
server_thread.daemon = True
server_thread.start()

time.sleep(3)

print("🚀 Flask server started successfully!")
print("📱 Dashboard is ready on port 5000")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🚀 Flask server started successfully!
📱 Dashboard is ready on port 5000


In [ ]:
import threading
import time

def start_flask_new():
    app.run(
        host="0.0.0.0",
        port=5001,
        debug=False,
        use_reloader=False
    )

server_thread = threading.Thread(target=start_flask_new)
server_thread.daemon = True
server_thread.start()

time.sleep(3)

print("🚀 New Flask server started!")
print("📱 New port: 5001")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


🚀 New Flask server started!
📱 New port: 5001


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(5001)

print("🌐 NEW DASHBOARD LINK:")
print(public_url)

In [ ]:
from pyngrok import ngrok

# Enter your ngrok token below
ngrok.set_auth_token("3KKU3LbrYRSsjH6LWrsYJBzaqiN_2UMMTtXXsQSgTPCF41fRn")

public_url = ngrok.connect(5000)

print("🌐 YOUR DASHBOARD LINK:")
print(public_url)

🌐 YOUR DASHBOARD LINK:
NgrokTunnel: "https://spider-dreamt-plated.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
run_task("Daily Backup")

print("\n📊 Execution History")
print("-" * 40)

for record in execution_history:
    print("Task:", record["task"])
    print("Status:", record["status"])
    print("Started:", record["started"])
    print("Finished:", record["finished"])
    print("-" * 40)

▶️ Running: Daily Backup
✅ Daily Backup completed!

📊 Execution History
----------------------------------------
Task: Daily Backup
Status: Success
Started: 2026-10-06 17:34:46.136164
Finished: 2026-10-06 17:34:46.136231
----------------------------------------


In [ ]:
print("🔄 Automatic monitoring is enabled!")
print("📊 Dashboard will refresh execution history every 5 seconds.")

🔄 Automatic monitoring is enabled!
📊 Dashboard will refresh execution history every 5 seconds.


In [ ]:
!git clone https://github.com/Pavithra-627/task-scheduler-job-runner.git

Cloning into 'task-scheduler-job-runner'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 19 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 8.27 KiB | 8.27 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
%cd task-scheduler-job-runner

/content/task-scheduler-job-runner


In [ ]:
!pip install -r requirements.txt -q

In [1]:
!ls -lh /content/*.ipynb

ls: cannot access '/content/*.ipynb': No such file or directory
